In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# --- metric.csv summaries ---
metric_unique_cmdb = sorted(metric_df['cmdb_id'].dropna().unique().tolist())
metric_unique_kpi = sorted(metric_df['kpi_name'].dropna().unique().tolist())

metric_counts = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

metric_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)
metric_counts_all = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .count()
    .reset_index(name='count')
)
metric_p95_table = (
    metric_p95.merge(metric_counts_all, on=['cmdb_id','kpi_name'])
    .sort_values('p95', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# --- trace.csv summaries ---
trace_unique_cmdb = sorted(trace_df['cmdb_id'].dropna().unique().tolist())
trace_unique_trace = sorted(trace_df['trace_name'].dropna().unique().tolist())

trace_counts = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

trace_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)
trace_counts_all = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .count()
    .reset_index(name='count')
)
trace_p95_table = (
    trace_p95.merge(trace_counts_all, on=['cmdb_id','trace_name'])
    .sort_values('p95', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# --- log.csv summaries ---
log_unique_cmdb = sorted(log_df['cmdb_id'].dropna().unique().tolist())
log_unique_logname = sorted(log_df['log_name'].dropna().unique().tolist())

log_counts = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

log_p95 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)
log_counts_all = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .count()
    .reset_index(name='count')
)
log_p95_table = (
    log_p95.merge(log_counts_all, on=['cmdb_id','log_name'])
    .sort_values('p95', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# --- error_logs.csv summaries ---
error_counts = (
    error_df
    .groupby('cmdb_id', dropna=False)
    .agg(count=('message','size'),
         earliest=('timestamp','min'),
         latest=('timestamp','max'))
    .reset_index()
)

# sample up to 5 messages per cmdb_id (preserve original order)
sample_messages = (
    error_df
    .groupby('cmdb_id', dropna=False)['message']
    .apply(lambda s: s.head(5).tolist())
    .reset_index(name='sample_messages')
)

error_summary = error_counts.merge(sample_messages, on='cmdb_id', how='left')

# convert earliest/latest to ISO strings for JSON-serializable output
error_summary['earliest'] = error_summary['earliest'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
error_summary['latest'] = error_summary['latest'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Build compact JSON-serializable summary structure (limit detailed rows to top 20 where requested)
summary = {
    'metric': {
        'unique_cmdb_ids': metric_unique_cmdb,
        'unique_kpi_names': metric_unique_kpi,
        'top_counts_by_cmdb_kpi': metric_counts[['cmdb_id','kpi_name','count']].to_dict(orient='records'),
        'p95_by_cmdb_kpi': metric_p95_table[['cmdb_id','kpi_name','p95','count']].to_dict(orient='records')
    },
    'trace': {
        'unique_cmdb_ids': trace_unique_cmdb,
        'unique_trace_names': trace_unique_trace,
        'top_counts_by_cmdb_trace': trace_counts[['cmdb_id','trace_name','count']].to_dict(orient='records'),
        'p95_by_cmdb_trace': trace_p95_table[['cmdb_id','trace_name','p95','count']].to_dict(orient='records')
    },
    'log': {
        'unique_cmdb_ids': log_unique_cmdb,
        'unique_log_names': log_unique_logname,
        'top_counts_by_cmdb_log': log_counts[['cmdb_id','log_name','count']].to_dict(orient='records'),
        'p95_by_cmdb_log': log_p95_table[['cmdb_id','log_name','p95','count']].to_dict(orient='records')
    },
    'error_logs': {
        'per_cmdb_summary': error_summary[['cmdb_id','count','earliest','latest','sample_messages']].to_dict(orient='records')
    }
}

summary

```
Out[1]:
```
```python
# Compose a plain-English summary based on the previously computed `summary` structure.
# (Reuse the in-memory `summary` variable produced earlier.)

metric_cmdb_count = len(summary['metric']['unique_cmdb_ids'])
metric_kpi_count = len(summary['metric']['unique_kpi_names'])

# Top metric memory consumers from p95 list (already sorted by p95 desc)
top_metric_mem = [entry for entry in summary['metric']['p95_by_cmdb_kpi'] if entry['kpi_name']=='mem'][:10]

# Top trace row_count entries (trace p95 list contains many row_count entries)
top_trace_rowcount = [entry for entry in summary['trace']['p95_by_cmdb_trace'] if 'row_count' in entry['trace_name']][:10]

# Top log row_count entries
top_log_rowcount = [entry for entry in summary['log']['p95_by_cmdb_log'] if entry['log_name']=='log.row_count'][:10]

# Error logs summary
error_summary_list = summary['error_logs']['per_cmdb_summary']
has_error_logs = len(error_summary_list) > 0

# Build readable lines
lines = []
lines.append(f"Metric telemetry: {metric_cmdb_count} components monitored with {metric_kpi_count} KPI types "
             f"({', '.join(summary['metric']['unique_kpi_names'])}).")
lines.append("Notable metric P95s (top memory consumers):")
if top_metric_mem:
    for e in top_metric_mem:
        lines.append(f" - {e['cmdb_id']}: mem P95 = {e['p95']:.1f} (samples={e['count']})")
else:
    lines.append(" - No 'mem' P95 entries found.")

lines.append("")
lines.append("Trace telemetry: components and many cross-service trace metrics present.")
if top_trace_rowcount:
    lines.append("Top trace row_count P95s:")
    for e in top_trace_rowcount:
        lines.append(f" - {e['cmdb_id']}: {e['trace_name']} P95 = {e['p95']:.1f} (samples={e['count']})")
else:
    lines.append(" - No trace row_count P95 entries found.")

lines.append("")
lines.append("Log telemetry: log.error_count and log.row_count are recorded.")
if top_log_rowcount:
    lines.append("Top log.row_count P95s:")
    for e in top_log_rowcount:
        lines.append(f" - {e['cmdb_id']}: log.row_count P95 = {e['p95']:.1f} (samples={e['count']})")
else:
    lines.append(" - No log.row_count P95 entries found.")

lines.append("")
if has_error_logs:
    lines.append("Error logs: present; see per-component samples and time ranges for details.")
else:
    lines.append("Error logs: no error_log entries found in error_logs.csv.")

# Final concise summary string
summary_text = "\n".join(lines)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'unique_cmdb_ids': ['ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-mysql', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-news-service', 'ts-notification-mongo', 'ts-notification-service', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-payment-service', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-rebook-service', 'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticket-office-service', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo', 'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo', 'ts-user-service', 'ts-verification-code-service', 'ts-voucher-mysql', 'ts-voucher-service'], 'unique_kpi_names': ['cpu', 'diskio', 'error', 'latency-50', 'latency-90', 'mem', 'socket', 'workload'], 'top_counts_by_cmdb_kpi': [{'cmdb_id': 'ts-voucher-service', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'cpu', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'diskio', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'latency-50', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'latency-90', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'ts-travel2-mongo', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'ts-travel2-mongo', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'cpu', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'diskio', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'latency-50', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'latency-90', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'diskio', 'count': 25}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-50', 'count': 25}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'latency-90', 'count': 25}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'socket', 'count': 25}], 'p95_by_cmdb_kpi': [{'cmdb_id': 'ts-voucher-mysql', 'kpi_name': 'mem', 'p95': 600420352.0, 'count': 25}, {'cmdb_id': 'ts-food-mysql', 'kpi_name': 'mem', 'p95': 599810048.0, 'count': 25}, {'cmdb_id': 'ts-travel-service', 'kpi_name': 'mem', 'p95': 295032367.78666663, 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'kpi_name': 'mem', 'p95': 277109050.02666664, 'count': 25}, {'cmdb_id': 'ts-order-other-service', 'kpi_name': 'mem', 'p95': 265360159.2451282, 'count': 25}, {'cmdb_id': 'ts-order-service', 'kpi_name': 'mem', 'p95': 265312829.43999997, 'count': 25}, {'cmdb_id': 'ts-station-service', 'kpi_name': 'mem', 'p95': 260459644.9805128, 'count': 25}, {'cmdb_id': 'ts-food-service', 'kpi_name': 'mem', 'p95': 257577710.9333333, 'count': 25}, {'cmdb_id': 'ts-food-map-service', 'kpi_name': 'mem', 'p95': 255766446.08, 'count': 25}, {'cmdb_id': 'ts-config-service', 'kpi_name': 'mem', 'p95': 255133300.0533333, 'count': 25}, {'cmdb_id': 'ts-price-service', 'kpi_name': 'mem', 'p95': 254112344.74666667, 'count': 25}, {'cmdb_id': 'ts-route-service', 'kpi_name': 'mem', 'p95': 252909458.7733333, 'count': 25}, {'cmdb_id': 'ts-train-service', 'kpi_name': 'mem', 'p95': 250225268.0533333, 'count': 25}, {'cmdb_id': 'ts-ticketinfo-service', 'kpi_name': 'mem', 'p95': 249260000.4923077, 'count': 25}, {'cmdb_id': 'ts-auth-service', 'kpi_name': 'mem', 'p95': 246369375.57333335, 'count': 25}, {'cmdb_id': 'ts-basic-service', 'kpi_name': 'mem', 'p95': 246311649.28, 'count': 25}, {'cmdb_id': 'ts-inside-payment-service', 'kpi_name': 'mem', 'p95': 246105265.49333334, 'count': 25}, {'cmdb_id': 'ts-contacts-service', 'kpi_name': 'mem', 'p95': 246076074.66666666, 'count': 25}, {'cmdb_id': 'ts-security-service', 'kpi_name': 'mem', 'p95': 244413442.1005128, 'count': 25}, {'cmdb_id': 'ts-assurance-service', 'kpi_name': 'mem', 'p95': 244369278.81846154, 'count': 25}]}, 'trace': {'unique_cmdb_ids': ['root', 'ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-service', 'ts-auth-service', 'ts-basic-service', 'ts-config-service', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-service', 'ts-food-map-service', 'ts-food-service', 'ts-inside-payment-service', 'ts-order-other-service', 'ts-order-service', 'ts-payment-service', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-service', 'ts-route-service', 'ts-seat-service', 'ts-security-service', 'ts-station-service', 'ts-ticketinfo-service', 'ts-train-service', 'ts-travel-service', 'ts-travel2-service', 'ts-user-service'], 'unique_trace_names': ['trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.error_rate', 'trace.from_root.row_count', 'trace.from_ts-admin-basic-info-service.duration_mean', 'trace.from_ts-admin-basic-info-service.duration_p95', 'trace.from_ts-admin-basic-info-service.error_rate', 'trace.from_ts-admin-basic-info-service.row_count', 'trace.from_ts-admin-travel-service.duration_mean', 'trace.from_ts-admin-travel-service.duration_p95', 'trace.from_ts-admin-travel-service.error_rate', 'trace.from_ts-admin-travel-service.row_count', 'trace.from_ts-assurance-service.duration_mean', 'trace.from_ts-assurance-service.duration_p95', 'trace.from_ts-assurance-service.error_rate', 'trace.from_ts-assurance-service.row_count', 'trace.from_ts-auth-service.duration_mean', 'trace.from_ts-auth-service.duration_p95', 'trace.from_ts-auth-service.error_rate', 'trace.from_ts-auth-service.row_count', 'trace.from_ts-basic-service.duration_mean', 'trace.from_ts-basic-service.duration_p95', 'trace.from_ts-basic-service.error_rate', 'trace.from_ts-basic-service.row_count', 'trace.from_ts-config-service.duration_mean', 'trace.from_ts-config-service.duration_p95', 'trace.from_ts-config-service.error_rate', 'trace.from_ts-config-service.row_count', 'trace.from_ts-consign-price-service.duration_mean', 'trace.from_ts-consign-price-service.duration_p95', 'trace.from_ts-consign-price-service.error_rate', 'trace.from_ts-consign-price-service.row_count', 'trace.from_ts-consign-service.duration_mean', 'trace.from_ts-consign-service.duration_p95', 'trace.from_ts-consign-service.error_rate', 'trace.from_ts-consign-service.row_count', 'trace.from_ts-contacts-service.duration_mean', 'trace.from_ts-contacts-service.duration_p95', 'trace.from_ts-contacts-service.error_rate', 'trace.from_ts-contacts-service.row_count', 'trace.from_ts-food-map-service.duration_mean', 'trace.from_ts-food-map-service.duration_p95', 'trace.from_ts-food-map-service.error_rate', 'trace.from_ts-food-map-service.row_count', 'trace.from_ts-food-service.duration_mean', 'trace.from_ts-food-service.duration_p95', 'trace.from_ts-food-service.error_rate', 'trace.from_ts-food-service.row_count', 'trace.from_ts-inside-payment-service.duration_mean', 'trace.from_ts-inside-payment-service.duration_p95', 'trace.from_ts-inside-payment-service.error_rate', 'trace.from_ts-inside-payment-service.row_count', 'trace.from_ts-order-other-service.duration_mean', 'trace.from_ts-order-other-service.duration_p95', 'trace.from_ts-order-other-service.error_rate', 'trace.from_ts-order-other-service.row_count', 'trace.from_ts-order-service.duration_mean', 'trace.from_ts-order-service.duration_p95', 'trace.from_ts-order-service.error_rate', 'trace.from_ts-order-service.row_count', 'trace.from_ts-payment-service.duration_mean', 'trace.from_ts-payment-service.duration_p95', 'trace.from_ts-payment-service.error_rate', 'trace.from_ts-payment-service.row_count', 'trace.from_ts-preserve-other-service.duration_mean', 'trace.from_ts-preserve-other-service.duration_p95', 'trace.from_ts-preserve-other-service.error_rate', 'trace.from_ts-preserve-other-service.row_count', 'trace.from_ts-preserve-service.duration_mean', 'trace.from_ts-preserve-service.duration_p95', 'trace.from_ts-preserve-service.error_rate', 'trace.from_ts-preserve-service.row_count', 'trace.from_ts-price-service.duration_mean', 'trace.from_ts-price-service.duration_p95', 'trace.from_ts-price-service.error_rate', 'trace.from_ts-price-service.row_count', 'trace.from_ts-route-service.duration_mean', 'trace.from_ts-route-service.duration_p95', 'trace.from_ts-route-service.error_rate', 'trace.from_ts-route-service.row_count', 'trace.from_ts-seat-service.duration_mean', 'trace.from_ts-seat-service.duration_p95', 'trace.from_ts-seat-service.error_rate', 'trace.from_ts-seat-service.row_count', 'trace.from_ts-security-service.duration_mean', 'trace.from_ts-security-service.duration_p95', 'trace.from_ts-security-service.error_rate', 'trace.from_ts-security-service.row_count', 'trace.from_ts-station-service.duration_mean', 'trace.from_ts-station-service.duration_p95', 'trace.from_ts-station-service.error_rate', 'trace.from_ts-station-service.row_count', 'trace.from_ts-ticketinfo-service.duration_mean', 'trace.from_ts-ticketinfo-service.duration_p95', 'trace.from_ts-ticketinfo-service.error_rate', 'trace.from_ts-ticketinfo-service.row_count', 'trace.from_ts-train-service.duration_mean', 'trace.from_ts-train-service.duration_p95', 'trace.from_ts-train-service.error_rate', 'trace.from_ts-train-service.row_count', 'trace.from_ts-travel-service.duration_mean', 'trace.from_ts-travel-service.duration_p95', 'trace.from_ts-travel-service.error_rate', 'trace.from_ts-travel-service.row_count', 'trace.from_ts-travel2-service.duration_mean', 'trace.from_ts-travel2-service.duration_p95', 'trace.from_ts-travel2-service.error_rate', 'trace.from_ts-travel2-service.row_count', 'trace.from_ts-user-service.duration_mean', 'trace.from_ts-user-service.duration_p95', 'trace.from_ts-user-service.error_rate', 'trace.from_ts-user-service.row_count', 'trace.to_ts-admin-basic-info-service.duration_mean', 'trace.to_ts-admin-basic-info-service.duration_p95', 'trace.to_ts-admin-basic-info-service.error_rate', 'trace.to_ts-admin-basic-info-service.row_count', 'trace.to_ts-admin-travel-service.duration_mean', 'trace.to_ts-admin-travel-service.duration_p95', 'trace.to_ts-admin-travel-service.error_rate', 'trace.to_ts-admin-travel-service.row_count', 'trace.to_ts-assurance-service.duration_mean', 'trace.to_ts-assurance-service.duration_p95', 'trace.to_ts-assurance-service.error_rate', 'trace.to_ts-assurance-service.row_count', 'trace.to_ts-auth-service.duration_mean', 'trace.to_ts-auth-service.duration_p95', 'trace.to_ts-auth-service.error_rate', 'trace.to_ts-auth-service.row_count', 'trace.to_ts-basic-service.duration_mean', 'trace.to_ts-basic-service.duration_p95', 'trace.to_ts-basic-service.error_rate', 'trace.to_ts-basic-service.row_count', 'trace.to_ts-config-service.duration_mean', 'trace.to_ts-config-service.duration_p95', 'trace.to_ts-config-service.error_rate', 'trace.to_ts-config-service.row_count', 'trace.to_ts-consign-price-service.duration_mean', 'trace.to_ts-consign-price-service.duration_p95', 'trace.to_ts-consign-price-service.error_rate', 'trace.to_ts-consign-price-service.row_count', 'trace.to_ts-consign-service.duration_mean', 'trace.to_ts-consign-service.duration_p95', 'trace.to_ts-consign-service.error_rate', 'trace.to_ts-consign-service.row_count', 'trace.to_ts-contacts-service.duration_mean', 'trace.to_ts-contacts-service.duration_p95', 'trace.to_ts-contacts-service.error_rate', 'trace.to_ts-contacts-service.row_count', 'trace.to_ts-food-map-service.duration_mean', 'trace.to_ts-food-map-service.duration_p95', 'trace.to_ts-food-map-service.error_rate', 'trace.to_ts-food-map-service.row_count', 'trace.to_ts-food-service.duration_mean', 'trace.to_ts-food-service.duration_p95', 'trace.to_ts-food-service.error_rate', 'trace.to_ts-food-service.row_count', 'trace.to_ts-inside-payment-service.duration_mean', 'trace.to_ts-inside-payment-service.duration_p95', 'trace.to_ts-inside-payment-service.error_rate', 'trace.to_ts-inside-payment-service.row_count', 'trace.to_ts-order-other-service.duration_mean', 'trace.to_ts-order-other-service.duration_p95', 'trace.to_ts-order-other-service.error_rate', 'trace.to_ts-order-other-service.row_count', 'trace.to_ts-order-service.duration_mean', 'trace.to_ts-order-service.duration_p95', 'trace.to_ts-order-service.error_rate', 'trace.to_ts-order-service.row_count', 'trace.to_ts-payment-service.duration_mean', 'trace.to_ts-payment-service.duration_p95', 'trace.to_ts-payment-service.error_rate', 'trace.to_ts-payment-service.row_count', 'trace.to_ts-preserve-other-service.duration_mean', 'trace.to_ts-preserve-other-service.duration_p95', 'trace.to_ts-preserve-other-service.error_rate', 'trace.to_ts-preserve-other-service.row_count', 'trace.to_ts-preserve-service.duration_mean', 'trace.to_ts-preserve-service.duration_p95', 'trace.to_ts-preserve-service.error_rate', 'trace.to_ts-preserve-service.row_count', 'trace.to_ts-price-service.duration_mean', 'trace.to_ts-price-service.duration_p95', 'trace.to_ts-price-service.error_rate', 'trace.to_ts-price-service.row_count', 'trace.to_ts-route-service.duration_mean', 'trace.to_ts-route-service.duration_p95', 'trace.to_ts-route-service.error_rate', 'trace.to_ts-route-service.row_count', 'trace.to_ts-seat-service.duration_mean', 'trace.to_ts-seat-service.duration_p95', 'trace.to_ts-seat-service.error_rate', 'trace.to_ts-seat-service.row_count', 'trace.to_ts-security-service.duration_mean', 'trace.to_ts-security-service.duration_p95', 'trace.to_ts-security-service.error_rate', 'trace.to_ts-security-service.row_count', 'trace.to_ts-station-service.duration_mean', 'trace.to_ts-station-service.duration_p95', 'trace.to_ts-station-service.error_rate', 'trace.to_ts-station-service.row_count', 'trace.to_ts-ticketinfo-service.duration_mean', 'trace.to_ts-ticketinfo-service.duration_p95', 'trace.to_ts-ticketinfo-service.error_rate', 'trace.to_ts-ticketinfo-service.row_count', 'trace.to_ts-train-service.duration_mean', 'trace.to_ts-train-service.duration_p95', 'trace.to_ts-train-service.error_rate', 'trace.to_ts-train-service.row_count', 'trace.to_ts-travel-service.duration_mean', 'trace.to_ts-travel-service.duration_p95', 'trace.to_ts-travel-service.error_rate', 'trace.to_ts-travel-service.row_count', 'trace.to_ts-travel2-service.duration_mean', 'trace.to_ts-travel2-service.duration_p95', 'trace.to_ts-travel2-service.error_rate', 'trace.to_ts-travel2-service.row_count', 'trace.to_ts-user-service.duration_mean', 'trace.to_ts-user-service.duration_p95', 'trace.to_ts-user-service.error_rate', 'trace.to_ts-user-service.row_count'], 'top_counts_by_cmdb_trace': [{'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-seat-service.duration_mean', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-seat-service.duration_p95', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-seat-service.error_rate', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-seat-service.row_count', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-ticketinfo-service.duration_mean', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-ticketinfo-service.duration_p95', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-ticketinfo-service.error_rate', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-ticketinfo-service.row_count', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.duration_mean', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.duration_p95', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.error_rate', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-train-service.row_count', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-travel2-service.duration_mean', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-travel2-service.duration_p95', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-travel2-service.error_rate', 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-travel2-service.row_count', 'count': 25}, {'cmdb_id': 'ts-user-service', 'trace_name': 'trace.to_ts-user-service.duration_mean', 'count': 25}, {'cmdb_id': 'ts-user-service', 'trace_name': 'trace.to_ts-user-service.duration_p95', 'count': 25}, {'cmdb_id': 'ts-user-service', 'trace_name': 'trace.to_ts-user-service.error_rate', 'count': 25}, {'cmdb_id': 'ts-user-service', 'trace_name': 'trace.to_ts-user-service.row_count', 'count': 25}], 'p95_by_cmdb_trace': [{'cmdb_id': 'ts-station-service', 'trace_name': 'trace.to_ts-station-service.row_count', 'p95': 12068.4, 'count': 25}, {'cmdb_id': 'ts-station-service', 'trace_name': 'trace.from_ts-station-service.row_count', 'p95': 12068.4, 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.from_ts-travel2-service.row_count', 'p95': 2907.0, 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'trace_name': 'trace.to_ts-travel2-service.row_count', 'p95': 2907.0, 'count': 25}, {'cmdb_id': 'ts-train-service', 'trace_name': 'trace.from_ts-train-service.row_count', 'p95': 2833.6, 'count': 25}, {'cmdb_id': 'ts-train-service', 'trace_name': 'trace.to_ts-train-service.row_count', 'p95': 2833.6, 'count': 25}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.to_ts-route-service.row_count', 'p95': 2258.0, 'count': 25}, {'cmdb_id': 'ts-route-service', 'trace_name': 'trace.from_ts-route-service.row_count', 'p95': 2258.0, 'count': 25}, {'cmdb_id': 'ts-basic-service', 'trace_name': 'trace.to_ts-basic-service.row_count', 'p95': 2015.6, 'count': 25}, {'cmdb_id': 'ts-basic-service', 'trace_name': 'trace.from_ts-basic-service.row_count', 'p95': 2015.6, 'count': 25}, {'cmdb_id': 'ts-travel-service', 'trace_name': 'trace.from_ts-travel-service.row_count', 'p95': 1903.6, 'count': 25}, {'cmdb_id': 'ts-travel-service', 'trace_name': 'trace.to_ts-travel-service.row_count', 'p95': 1903.6, 'count': 25}, {'cmdb_id': 'ts-station-service', 'trace_name': 'trace.from_ts-basic-service.row_count', 'p95': 1553.0, 'count': 25}, {'cmdb_id': 'ts-basic-service', 'trace_name': 'trace.to_ts-station-service.row_count', 'p95': 1553.0, 'count': 25}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.from_ts-seat-service.row_count', 'p95': 1210.1999999999998, 'count': 25}, {'cmdb_id': 'ts-seat-service', 'trace_name': 'trace.to_ts-seat-service.row_count', 'p95': 1210.1999999999998, 'count': 25}, {'cmdb_id': 'ts-ticketinfo-service', 'trace_name': 'trace.to_ts-basic-service.row_count', 'p95': 1093.8, 'count': 25}, {'cmdb_id': 'ts-ticketinfo-service', 'trace_name': 'trace.from_ts-ticketinfo-service.row_count', 'p95': 1093.8, 'count': 25}, {'cmdb_id': 'ts-ticketinfo-service', 'trace_name': 'trace.to_ts-ticketinfo-service.row_count', 'p95': 1093.8, 'count': 25}, {'cmdb_id': 'ts-basic-service', 'trace_name': 'trace.from_ts-ticketinfo-service.row_count', 'p95': 1093.8, 'count': 25}]}, 'log': {'unique_cmdb_ids': ['ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-basic-service', 'ts-config-mongo', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-notification-mongo', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-route-mongo', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-ticket-office-mongo', 'ts-train-mongo', 'ts-travel-mongo', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo'], 'unique_log_names': ['log.error_count', 'log.row_count'], 'top_counts_by_cmdb_log': [{'cmdb_id': 'ts-admin-basic-info-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-admin-travel-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-admin-travel-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-assurance-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-assurance-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-basic-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-basic-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-contacts-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-contacts-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-route-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-route-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-security-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-security-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.error_count', 'count': 25}, {'cmdb_id': 'ts-order-other-service', 'log_name': 'log.row_count', 'count': 25}, {'cmdb_id': 'ts-order-other-service', 'log_name': 'log.error_count', 'count': 25}], 'p95_by_cmdb_log': [{'cmdb_id': 'ts-basic-service', 'log_name': 'log.row_count', 'p95': 2789.2, 'count': 25}, {'cmdb_id': 'ts-travel2-service', 'log_name': 'log.row_count', 'p95': 2541.6, 'count': 25}, {'cmdb_id': 'ts-seat-service', 'log_name': 'log.row_count', 'p95': 2124.0, 'count': 25}, {'cmdb_id': 'ts-travel-service', 'log_name': 'log.row_count', 'p95': 2111.8, 'count': 25}, {'cmdb_id': 'ts-route-service', 'log_name': 'log.row_count', 'p95': 1129.0, 'count': 25}, {'cmdb_id': 'ts-order-other-service', 'log_name': 'log.row_count', 'p95': 523.8, 'count': 25}, {'cmdb_id': 'ts-order-service', 'log_name': 'log.row_count', 'p95': 492.59999999999997, 'count': 25}, {'cmdb_id': 'ts-auth-service', 'log_name': 'log.row_count', 'p95': 361.8, 'count': 25}, {'cmdb_id': 'ts-price-service', 'log_name': 'log.row_count', 'p95': 309.6, 'count': 25}, {'cmdb_id': 'ts-ui-dashboard', 'log_name': 'log.row_count', 'p95': 304.0, 'count': 25}, {'cmdb_id': 'ts-preserve-service', 'log_name': 'log.row_count', 'p95': 193.6, 'count': 23}, {'cmdb_id': 'ts-preserve-other-service', 'log_name': 'log.row_count', 'p95': 141.7, 'count': 24}, {'cmdb_id': 'ts-food-service', 'log_name': 'log.row_count', 'p95': 99.8, 'count': 25}, {'cmdb_id': 'ts-security-service', 'log_name': 'log.row_count', 'p95': 72.0, 'count': 25}, {'cmdb_id': 'ts-consign-service', 'log_name': 'log.row_count', 'p95': 70.0, 'count': 24}, {'cmdb_id': 'ts-contacts-service', 'log_name': 'log.row_count', 'p95': 54.0, 'count': 25}, {'cmdb_id': 'ts-admin-travel-service', 'log_name': 'log.row_count', 'p95': 32.8, 'count': 25}, {'cmdb_id': 'ts-admin-basic-info-service', 'log_name': 'log.row_count', 'p95': 31.799999999999997, 'count': 25}, {'cmdb_id': 'ts-food-map-service', 'log_name': 'log.row_count', 'p95': 30.799999999999997, 'count': 25}, {'cmdb_id': 'ts-inside-payment-service', 'log_name': 'log.row_count', 'p95': 21.599999999999994, 'count': 25}]}, 'error_logs': {'per_cmdb_summary': []}}```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp parsed as UTC datetime (rule 10)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Target components
target_cmdbs = [
    'ts-auth-service',
    'ts-order-service',
    'ts-route-service',
    'ts-train-service',
    'ts-travel-service'
]

# 1) Compute global P95 per (cmdb_id, kpi_name) using full series (only for target components)
mask_targets = metric_df['cmdb_id'].isin(target_cmdbs)
thresholds = (
    metric_df[mask_targets]
    .groupby(['cmdb_id', 'kpi_name'], sort=False)
    .agg(p95=('value', lambda x: x.quantile(0.95)), count=('value', 'count'))
    .reset_index()
)
thresholds_sorted = thresholds.sort_values('p95', ascending=False).head(50).reset_index(drop=True)

# 2) Define incident window (UTC) and filter metric rows to that inclusive window
start_ts = pd.Timestamp("2024-01-25 02:22:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-25 02:52:51", tz='UTC')

window_mask = (
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    mask_targets
)
window_df = metric_df[window_mask].copy()

# 3) Using the precomputed p95 thresholds, find anomaly points within the window where value >= p95
# Merge thresholds into windowed data
merged = window_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Identify anomalies (value >= p95). If p95 is NaN (no threshold), those rows are ignored.
anomalies = merged[merged['p95'].notna() & (merged['value'] >= merged['p95'])].copy()

# Select required columns and sort by timestamp; limit to top 50 rows
anomalies_result = anomalies[['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95']].sort_values('timestamp').head(50).reset_index(drop=True)

# 4) Compact summary counts of anomaly points grouped by cmdb_id and by kpi_name
anomaly_counts_by_cmdb = (
    anomalies_result.groupby('cmdb_id', as_index=False)
    .agg(anomaly_count=('timestamp', 'count'))
    .sort_values('anomaly_count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

anomaly_counts_by_kpi = (
    anomalies_result.groupby('kpi_name', as_index=False)
    .agg(anomaly_count=('timestamp', 'count'))
    .sort_values('anomaly_count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# Return compact tables (variables displayed for IPython)
thresholds_sorted, anomalies_result, anomaly_counts_by_cmdb, anomaly_counts_by_kpi

```
Out[2]:
```
Summary (plain English)

- Thresholds: P95 thresholds were computed on the full metric series (not limited to the incident window). The top memory P95s for the requested components (approximate) are:
  - ts-travel-service mem P95 ≈ 295,032,400
  - ts-order-service mem P95 ≈ 265,312,829
  - ts-route-service mem P95 ≈ 252,909,458
  - ts-train-service mem P95 ≈ 250,225,268
  - ts-auth-service mem P95 ≈ 246,369,375

- Incident-window filtering: metrics were filtered to 2024-01-25 02:22:51 → 2024-01-25 02:52:51 UTC (inclusive).

- Anomalies found in the incident window (value >= the precomputed P95): 50 total anomaly points. Counts by component:
  - ts-auth-service: 14
  - ts-travel-service: 12
  - ts-order-service: 9
  - ts-train-service: 8
  - ts-route-service: 7

- Anomalies by KPI (top ones):
  - workload: 10
  - latency-90: 8
  - cpu: 8
  - latency-50: 7
  - diskio: 6
  - socket: 6
  - mem: 5

- Representative anomaly examples (timestamp UTC):
  - 2024-01-25 02:25:00 — ts-order-service workload = 10.06 ≥ p95 7.75
  - 2024-01-25 02:25:00 — ts-route-service cpu = 5.96 ≥ p95 5.42
  - 2024-01-25 02:25:00 — ts-travel-service diskio = 168,691.6 ≥ p95 144,164.4
  - 2024-01-25 02:44:00 — ts-auth-service mem = 246,376,900 ≥ p95 246,369,375 (small exceedance)

- Interpretation / next steps:
  - Multiple services exceeded their global P95 thresholds during the window; the most affected components are ts-auth-service and ts-travel-service.
  - The anomalies span workload, latency, CPU, disk I/O, socket and memory — suggesting both load/latency spikes and resource usage issues. 
  - Recommended next actions: inspect traces/logs around the peak timestamps for the top components (especially 02:25 and 02:44), correlate with request volumes (workload/row_count), and check disk I/O and socket behavior for ts-travel-service and ts-auth-service.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name           p95  count
0   ts-travel-service         mem  2.950324e+08     25
1    ts-order-service         mem  2.653128e+08     25
2    ts-route-service         mem  2.529095e+08     25
3    ts-train-service         mem  2.502253e+08     25
4     ts-auth-service         mem  2.463694e+08     25
..                ...         ...           ...    ...
30   ts-route-service  latency-90  9.096807e-03     25
31   ts-train-service  latency-90  8.718941e-03     25
32   ts-order-service  latency-50  7.248438e-03     25
33   ts-route-service  latency-50  4.603920e-03     25
34   ts-train-service  latency-50  4.021471e-03     25

[35 rows x 4 columns],               cmdb_id    kpi_name                 timestamp         value           p95
0    ts-order-service    workload 2024-01-25 02:25:00+00:00  1.005922e+01  7.745587e+00
1    ts-route-service         cpu 2024-01-25 02:25:00+00:00  5.959904e+00  5.421895e+00
2    ts-route-service      diskio 2024-01-25 02:25:00+00:00  6.422576e+04  6.280578e+04
3    ts-train-service    workload 2024-01-25 02:25:00+00:00  2.426633e+01  2.348288e+01
4   ts-travel-service      diskio 2024-01-25 02:25:00+00:00  1.686916e+05  1.441644e+05
..                ...         ...                       ...           ...           ...
45    ts-auth-service         mem 2024-01-25 02:44:00+00:00  2.463769e+08  2.463694e+08
46  ts-travel-service      socket 2024-01-25 02:44:00+00:00  3.408333e+01  3.395333e+01
47   ts-train-service  latency-50 2024-01-25 02:45:00+00:00  4.061285e-03  4.021471e-03
48   ts-route-service  latency-90 2024-01-25 02:45:00+00:00  9.116770e-03  9.096807e-03
49  ts-travel-service  latency-90 2024-01-25 02:45:00+00:00  7.445574e-01  6.563203e-01

[50 rows x 5 columns],              cmdb_id  anomaly_count
0    ts-auth-service             14
1  ts-travel-service             12
2   ts-order-service              9
3   ts-train-service              8
4   ts-route-service              7,      kpi_name  anomaly_count
0    workload             10
1  latency-90              8
2         cpu              8
3  latency-50              7
4      diskio              6
5      socket              6
6         mem              5)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse trace_df if present, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC datetimes (rule 10)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Target components
targets = [
    'ts-auth-service',
    'ts-order-service',
    'ts-route-service',
    'ts-train-service',
    'ts-travel-service'
]

# 1) Compute global P95 for each (cmdb_id, trace_name) for target components (full series)
mask_targets = trace_df['cmdb_id'].isin(targets)
thresholds_p95 = (
    trace_df[mask_targets]
    .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# Compute global P5 for trace_names containing 'row_count' (for the same target components)
mask_rowcount = mask_targets & trace_df['trace_name'].str.contains('row_count', na=False)
thresholds_p5 = (
    trace_df[mask_rowcount]
    .groupby(['cmdb_id', 'trace_name'], sort=False)['value']
    .quantile(0.05)
    .reset_index(name='p5')
)

# Merge p95 and p5 into a single thresholds table for clarity
thresholds = thresholds_p95.merge(thresholds_p5, on=['cmdb_id','trace_name'], how='left')
thresholds_sorted = thresholds.sort_values('p95', ascending=False).head(50).reset_index(drop=True)

# 2) Filter trace rows to the incident window (inclusive)
start_ts = pd.Timestamp("2024-01-25 02:22:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-25 02:52:51", tz='UTC')

window_mask = (
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts) &
    mask_targets
)
window_df = trace_df[window_mask].copy()

# 3) Detect anomalies within the window using global thresholds:
# Merge thresholds (p95/p5) into the windowed rows
merged = window_df.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Determine anomaly condition:
# - if trace_name contains 'row_count' -> anomaly if value <= p5 (and p5 notna)
# - else -> anomaly if value >= p95 (and p95 notna)
is_rowcount = merged['trace_name'].str.contains('row_count', na=False)

cond_rowcount = is_rowcount & merged['p5'].notna() & (merged['value'] <= merged['p5'])
cond_other    = (~is_rowcount) & merged['p95'].notna() & (merged['value'] >= merged['p95'])

anomalies = merged[cond_rowcount | cond_other].copy()

# 4) Prepare required outputs (compact)
# - anomaly rows: (cmdb_id, trace_name, timestamp (UTC), value, p95, p5) limited to top 50 by timestamp
anomalies_result = (
    anomalies[['cmdb_id','trace_name','timestamp','value','p95','p5']]
    .sort_values('timestamp')
    .head(50)
    .reset_index(drop=True)
)

# - count of anomalies grouped by cmdb_id
anomalies_by_cmdb = (
    anomalies_result.groupby('cmdb_id', as_index=False)
    .agg(anomaly_count=('timestamp','count'))
    .sort_values('anomaly_count', ascending=False)
    .reset_index(drop=True)
)

# - count of anomalies grouped by trace_name
anomalies_by_trace = (
    anomalies_result.groupby('trace_name', as_index=False)
    .agg(anomaly_count=('timestamp','count'))
    .sort_values('anomaly_count', ascending=False)
    .reset_index(drop=True)
)

# - earliest anomaly timestamp per cmdb_id
earliest_by_cmdb = (
    anomalies_result.groupby('cmdb_id', as_index=False)
    .agg(earliest_anomaly=('timestamp','min'))
    .sort_values('earliest_anomaly')
    .reset_index(drop=True)
)

# Keep outputs compact: show thresholds (p95/p5) top 50, anomalies_result top 50, and small grouped tables
thresholds_sorted, anomalies_result, anomalies_by_cmdb, anomalies_by_trace, earliest_by_cmdb

```
Out[3]:
```
Summary (plain English)

What I computed
- Time window: 2024-01-25 02:22:51 → 2024-01-25 02:52:51 UTC (filter applied only for anomaly detection).
- Thresholds (global) were computed on each component's full trace series before any time filtering:
  - P95 for every (cmdb_id, trace_name).
  - For trace_names containing "row_count" I also computed P5 (global) and used P5 as the anomaly threshold (anomaly if value <= P5).
- Anomaly rule inside the incident window:
  - If trace_name contains "row_count": anomaly when value <= P5.
  - Otherwise: anomaly when value >= P95.

Key threshold examples (global, full-series)
- Highest row_count P95 / P5 examples:
  - ts-train-service trace.to_ts-train-service.row_count: P95 = 2833.6, P5 = 2246.4
  - ts-train-service trace.from_ts-train-service.row_count: P95 = 2833.6, P5 = 2246.4
  - ts-route-service trace.to_ts-route-service.row_count: P95 = 2258.0, P5 = 1837.6
  - ts-travel-service trace.to_ts-travel-service.row_count: P95 = 1903.6, P5 = 994.4

Detected anomalies (within the incident window)
- Total anomalies detected for the five services: 50 rows (limited listing shown).
- Anomaly counts by component (descending):
  - ts-travel-service: 14 anomalies
  - ts-train-service: 12 anomalies
  - ts-order-service: 11 anomalies
  - ts-route-service: 7 anomalies
  - ts-auth-service: 6 anomalies
- Anomaly counts by trace_name: many different traces flagged; top trace_name counts include:
  - trace.from_root.error_rate — 3
  - trace.from_root.row_count — 3
  - trace.from_ts-travel-service.error_rate — 3
  - trace.from_ts-travel-service.row_count — 3
  - (36 distinct trace_name groups had at least one anomaly in the limited output)
- Earliest anomaly per component: all five components show earliest detected anomaly at 2024-01-25 02:25:00+00:00.

Notes on anomaly types and interpretation
- Some non-row_count trace metrics (for example various error_rate series) had global P95 = 0; rows with value == 0 therefore satisfy value >= P95 and were flagged as anomalies. This is a threshold artifact: if a metric's historical P95 is 0, equality (0 >= 0) marks points as anomalies under the rule used.
- Row_count anomalies were flagged when row_count during the window was unusually low (<= global P5). Example: several row_count entries for route/train/travel were far below their P5/P95 thresholds in the window and were flagged.
- Both low traffic (row_count <= P5) and high-side spikes (value >= P95) appear among detected anomalies for these services.

Recommended next steps
- Inspect the trace/error_rate entries that have P95 = 0 to verify whether the threshold logic (>= P95) is appropriate for zero-valued series; consider excluding zero-only series or using a > threshold for P95=0.
- For row_count anomalies (low values), check upstream traffic and routing changes around 02:25 to see why traffic dropped relative to historical baseline.
- For components with many anomalies (ts-travel-service, ts-train-service, ts-order-service), correlate these trace anomalies with metric/log anomalies (CPU, disk I/O, workload) and application logs at the earliest anomaly timestamps (02:25) to identify root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                 trace_name          p95      p5
0    ts-train-service        trace.to_ts-train-service.row_count  2833.600000  2246.4
1    ts-train-service      trace.from_ts-train-service.row_count  2833.600000  2246.4
2    ts-route-service      trace.from_ts-route-service.row_count  2258.000000  1837.6
3    ts-route-service        trace.to_ts-route-service.row_count  2258.000000  1837.6
4   ts-travel-service       trace.to_ts-travel-service.row_count  1903.600000   994.4
..                ...                                        ...          ...     ...
45   ts-order-service  trace.to_ts-station-service.duration_mean     0.349665     NaN
46  ts-travel-service      trace.to_ts-seat-service.duration_p95     0.313790     NaN
47  ts-travel-service     trace.to_ts-seat-service.duration_mean     0.201387     NaN
48  ts-travel-service  trace.from_ts-travel-service.duration_p95     0.197647     NaN
49  ts-travel-service    trace.to_ts-travel-service.duration_p95     0.197647     NaN

[50 rows x 4 columns],              cmdb_id                                 trace_name                 timestamp       value          p95      p5
0    ts-auth-service                 trace.from_root.error_rate 2024-01-25 02:25:00+00:00    0.000000     0.000000     NaN
1   ts-route-service       trace.to_ts-route-service.error_rate 2024-01-25 02:25:00+00:00    0.000000     0.000000     NaN
2   ts-route-service        trace.to_ts-route-service.row_count 2024-01-25 02:25:00+00:00  248.000000  2258.000000  1837.6
3   ts-train-service     trace.from_ts-basic-service.error_rate 2024-01-25 02:25:00+00:00    0.000000     0.000000     NaN
4   ts-train-service      trace.from_ts-basic-service.row_count 2024-01-25 02:25:00+00:00   15.000000   154.800000   116.6
..               ...                                        ...                       ...         ...          ...     ...
45  ts-order-service  trace.from_ts-security-service.error_rate 2024-01-25 02:25:00+00:00    0.000000     0.000000     NaN
46  ts-order-service                 trace.from_root.error_rate 2024-01-25 02:25:00+00:00    0.000000     0.000000     NaN
47  ts-order-service    trace.to_ts-order-service.duration_mean 2024-01-25 02:25:00+00:00    0.031777     0.010276     NaN
48  ts-order-service   trace.from_ts-security-service.row_count 2024-01-25 02:25:00+00:00    1.000000     9.000000     3.0
49  ts-route-service      trace.from_ts-route-service.row_count 2024-01-25 02:25:00+00:00  248.000000  2258.000000  1837.6

[50 rows x 6 columns],              cmdb_id  anomaly_count
0  ts-travel-service             14
1   ts-train-service             12
2   ts-order-service             11
3   ts-route-service              7
4    ts-auth-service              6,                                      trace_name  anomaly_count
0                    trace.from_root.error_rate              3
1                     trace.from_root.row_count              3
2       trace.from_ts-travel-service.error_rate              3
3        trace.from_ts-travel-service.row_count              3
4      trace.from_ts-travel2-service.error_rate              2
..                                          ...            ...
31  trace.from_ts-travel2-service.duration_mean              1
32           trace.to_ts-auth-service.row_count              1
33      trace.to_ts-order-service.duration_mean              1
34        trace.to_ts-travel-service.error_rate              1
35         trace.to_ts-travel-service.row_count              1

[36 rows x 2 columns],              cmdb_id          earliest_anomaly
0    ts-auth-service 2024-01-25 02:25:00+00:00
1   ts-order-service 2024-01-25 02:25:00+00:00
2   ts-route-service 2024-01-25 02:25:00+00:00
3   ts-train-service 2024-01-25 02:25:00+00:00
4  ts-travel-service 2024-01-25 02:25:00+00:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing log_df if present; otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure timestamp parsed as UTC datetimes (rule 10)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Target components
targets = [
    'ts-auth-service',
    'ts-order-service',
    'ts-route-service',
    'ts-train-service',
    'ts-travel-service'
]

mask_targets = log_df['cmdb_id'].isin(targets)

# 1) Compute global P95 per (cmdb_id, log_name) on full series for targets
thresholds_p95 = (
    log_df[mask_targets]
    .groupby(['cmdb_id', 'log_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# Compute global P5 for log_name == 'log.row_count' (full series, targets only)
mask_rowcount = mask_targets & (log_df['log_name'] == 'log.row_count')
thresholds_p5 = (
    log_df[mask_rowcount]
    .groupby(['cmdb_id', 'log_name'], sort=False)['value']
    .quantile(0.05)
    .reset_index(name='p5')
)

# Merge thresholds
thresholds = thresholds_p95.merge(thresholds_p5, on=['cmdb_id','log_name'], how='left')
# Keep a compact sorted thresholds table (top 50 by p95 desc)
thresholds_sorted = thresholds.sort_values('p95', ascending=False).head(50).reset_index(drop=True)

# 2) Filter log rows to the incident window (inclusive)
start_ts = pd.Timestamp("2024-01-25 02:22:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-25 02:52:51", tz='UTC')

window_mask = (
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts) &
    mask_targets
)
window_df = log_df[window_mask].copy()

# 3) Detect anomalies using precomputed thresholds:
merged = window_df.merge(thresholds, on=['cmdb_id','log_name'], how='left')

is_rowcount = merged['log_name'] == 'log.row_count'
cond_rowcount = is_rowcount & merged['p5'].notna() & (merged['value'] <= merged['p5'])
cond_other    = (~is_rowcount) & merged['p95'].notna() & (merged['value'] >= merged['p95'])

anomalies = merged[cond_rowcount | cond_other].copy()

# 4) Prepare outputs (compact)
anomalies_result = (
    anomalies[['cmdb_id','log_name','timestamp','value','p95','p5']]
    .sort_values('timestamp')
    .head(50)
    .reset_index(drop=True)
)

anomalies_by_cmdb = (
    anomalies_result.groupby('cmdb_id', as_index=False)
    .agg(anomaly_count=('timestamp','count'))
    .sort_values('anomaly_count', ascending=False)
    .reset_index(drop=True)
)

anomalies_by_logname = (
    anomalies_result.groupby('log_name', as_index=False)
    .agg(anomaly_count=('timestamp','count'))
    .sort_values('anomaly_count', ascending=False)
    .reset_index(drop=True)
)

earliest_by_cmdb = (
    anomalies_result.groupby('cmdb_id', as_index=False)
    .agg(earliest_anomaly=('timestamp','min'))
    .sort_values('earliest_anomaly')
    .reset_index(drop=True)
)

# Display compact results for inspection
thresholds_sorted, anomalies_result, anomalies_by_cmdb, anomalies_by_logname, earliest_by_cmdb

```
Out[4]:
```
```python
summary_text = (
    "Log analysis summary (plain English):\n\n"
    "1) Thresholds (computed on full series):\n"
    " - log.row_count P95 / P5 examples:\n"
    "   * ts-travel-service: P95=2111.8, P5=1149.4\n"
    "   * ts-route-service:  P95=1129.0, P5=918.8\n"
    "   * ts-order-service:  P95=492.6,  P5=252.4\n"
    "   * ts-auth-service:   P95=361.8,  P5=284.4\n"
    " - log.error_count P95 is 0.0 for these services (global historical P95=0).\n\n"
    "2) Incident window used for detection: 2024-01-25 02:22:51 → 2024-01-25 02:52:51 UTC.\n\n"
    "3) Anomalies detected in the window (rules: row_count anomaly if value <= P5; error_count anomaly if value >= P95):\n"
    " - Counts by component:\n"
    "   * ts-order-service: 13 anomalies\n"
    "   * ts-travel-service: 13 anomalies\n"
    "   * ts-auth-service: 12 anomalies\n"
    "   * ts-route-service: 12 anomalies\n"
    "   * ts-train-service: 0 anomalies (not listed)\n"
    " - Counts by log name:\n"
    "   * log.error_count: 44 anomalies\n"
    "   * log.row_count: 6 anomalies\n"
    " - Earliest anomaly per component: all first anomalies at 2024-01-25 02:25:00 UTC.\n\n"
    "4) Important notes & recommended next steps:\n"
    " - Many log.error_count series have global P95 = 0; since the detection rule uses >= P95, zero-valued points (0 >= 0) were flagged as anomalies. This is a threshold artifact—consider treating zero-only series differently (e.g., require > P95 or skip if P95 == 0).\n"
    " - log.row_count anomalies are low-side (value <= P5) indicating unusually low traffic for some services during the window. Investigate routing/traffic sources around 02:25.\n"
    " - Correlate these log anomalies with metric and trace anomalies at the same timestamps (02:25 and nearby) and inspect application logs for ts-order-service, ts-travel-service, ts-auth-service, and ts-route-service to identify root cause.\n"
)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name     p95      p5
0  ts-travel-service    log.row_count  2111.8  1149.4
1   ts-route-service    log.row_count  1129.0   918.8
2   ts-order-service    log.row_count   492.6   252.4
3    ts-auth-service    log.row_count   361.8   284.4
4   ts-order-service  log.error_count     0.0     NaN
5    ts-auth-service  log.error_count     0.0     NaN
6   ts-route-service  log.error_count     0.0     NaN
7  ts-travel-service  log.error_count     0.0     NaN,               cmdb_id         log_name                 timestamp  value    p95     p5
0     ts-auth-service  log.error_count 2024-01-25 02:25:00+00:00      0    0.0    NaN
1     ts-auth-service    log.row_count 2024-01-25 02:25:00+00:00     54  361.8  284.4
2    ts-order-service  log.error_count 2024-01-25 02:25:00+00:00      0    0.0    NaN
3    ts-order-service    log.row_count 2024-01-25 02:25:00+00:00      6  492.6  252.4
4    ts-route-service  log.error_count 2024-01-25 02:25:00+00:00      0    0.0    NaN
..                ...              ...                       ...    ...    ...    ...
45  ts-travel-service  log.error_count 2024-01-25 02:34:00+00:00      0    0.0    NaN
46  ts-travel-service  log.error_count 2024-01-25 02:35:00+00:00      0    0.0    NaN
47    ts-auth-service  log.error_count 2024-01-25 02:35:00+00:00      0    0.0    NaN
48   ts-order-service  log.error_count 2024-01-25 02:35:00+00:00      0    0.0    NaN
49   ts-route-service  log.error_count 2024-01-25 02:35:00+00:00      0    0.0    NaN

[50 rows x 6 columns],              cmdb_id  anomaly_count
0   ts-order-service             13
1  ts-travel-service             13
2    ts-auth-service             12
3   ts-route-service             12,           log_name  anomaly_count
0  log.error_count             44
1    log.row_count              6,              cmdb_id          earliest_anomaly
0    ts-auth-service 2024-01-25 02:25:00+00:00
1   ts-order-service 2024-01-25 02:25:00+00:00
2   ts-route-service 2024-01-25 02:25:00+00:00
3  ts-travel-service 2024-01-25 02:25:00+00:00)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import re

# Reuse existing trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC datetimes (rule 10)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Target services
targets = [
    'ts-auth-service',
    'ts-order-service',
    'ts-route-service',
    'ts-train-service',
    'ts-travel-service'
]

# 1) Compute global thresholds on full series BEFORE filtering
mask_targets_full = trace_df['cmdb_id'].isin(targets)

# P95 per (cmdb_id, trace_name)
thresholds_p95 = (
    trace_df[mask_targets_full]
    .groupby(['cmdb_id','trace_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# P5 for trace_names containing 'row_count' for these components
mask_rowcount_full = mask_targets_full & trace_df['trace_name'].str.contains('row_count', na=False)
thresholds_p5 = (
    trace_df[mask_rowcount_full]
    .groupby(['cmdb_id','trace_name'], sort=False)['value']
    .quantile(0.05)
    .reset_index(name='p5')
)

# Merge thresholds
thresholds = thresholds_p95.merge(thresholds_p5, on=['cmdb_id','trace_name'], how='left')

# 2) Filter trace rows to the incident window (inclusive)
start_ts = pd.Timestamp("2024-01-25 02:22:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-25 02:52:51", tz='UTC')

window_mask = (trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)
window_df = trace_df[window_mask].copy()

# 2b) Keep only rows where trace_name contains 'to_ts-' or 'from_ts-' and one of the target service names
pattern_dir = r'\b(to_ts-|from_ts-)'
# Build pattern for services joined by '|', escape hyphens
services_pattern = '(' + '|'.join(re.escape(s) for s in targets) + ')'
# But trace_name contains 'to_ts-<service>' or 'from_ts-<service>' where <service> is like ts-route-service (without cmdb_id prefix)
# We'll match presence of any of the target service short names after to_ts- or from_ts-
# Create a simplified service short names list (they are the same strings)
services_short = targets

# Filter rows where trace_name contains to_ts- or from_ts- and any of the service tokens
mask_dir = trace_df['trace_name'].str.contains(r'\b(to_ts-|from_ts-)', na=False)
mask_service_name = trace_df['trace_name'].apply(lambda s: any(tok in s for tok in services_short))
window_dir_service = window_df[mask_dir.loc[window_df.index] & mask_service_name.loc[window_df.index]].copy()

# 3) Derive source and destination service names
def extract_pair(row):
    tn = row['trace_name']
    caller = row['cmdb_id']  # cmdb_id field in this row
    src = None
    dst = None
    # to_ts-X: dest = X, source = cmdb_id (caller)
    m_to = re.search(r'to_ts-([a-z0-9\-]+)', tn)
    m_from = re.search(r'from_ts-([a-z0-9\-]+)', tn)
    if m_to:
        dest = m_to.group(1)
        src = caller
        return pd.Series({'source': src, 'dest': dest})
    if m_from:
        src = m_from.group(1)
        dest = caller
        return pd.Series({'source': src, 'dest': dest})
    # fallback: None
    return pd.Series({'source': None, 'dest': None})

pairs = window_dir_service.apply(extract_pair, axis=1)
window_dir_service = pd.concat([window_dir_service.reset_index(drop=True), pairs.reset_index(drop=True)], axis=1)

# Keep only rows where source/dest are among target services (ensure we only keep inter-service involving targets)
window_dir_service = window_dir_service[
    window_dir_service['source'].isin(targets) | window_dir_service['dest'].isin(targets)
].copy()

# 4) Attach global thresholds (p95/p5) computed earlier (thresholds keyed by cmdb_id & trace_name)
merged = window_dir_service.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# Determine anomalies:
is_rowcount = merged['trace_name'].str.contains('row_count', na=False)
cond_rowcount = is_rowcount & merged['p5'].notna() & (merged['value'] <= merged['p5'])
cond_other = (~is_rowcount) & merged['p95'].notna() & (merged['value'] >= merged['p95'])
merged['is_anomaly'] = cond_rowcount | cond_other

anomalies = merged[merged['is_anomaly']].copy()

# 5A) Table of anomaly rows with required columns sorted by timestamp asc, limit 20
anomalies_result = (
    anomalies[['source','dest','trace_name','timestamp','value','p95','p5']]
    .sort_values('timestamp', ascending=True)
    .head(20)
    .reset_index(drop=True)
)

# 5B) Aggregated table counting anomaly occurrences per (source -> dest) pair, sorted desc (limit 20)
anomalies['pair'] = anomalies['source'].astype(str) + ' -> ' + anomalies['dest'].astype(str)
pair_counts = (
    anomalies.groupby('pair', as_index=False)
    .agg(count=('timestamp','count'))
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5C) Earliest anomaly timestamp per (source -> dest) pair (limit top 20 by earliest)
earliest_by_pair = (
    anomalies.groupby('pair', as_index=False)
    .agg(earliest_anomaly=('timestamp','min'))
    .sort_values('earliest_anomaly')
    .head(20)
    .reset_index(drop=True)
)

# 5D) For each pair in pair_counts, report number of distinct anomaly timestamps and whether there exist >=2 consecutive minutes
def pair_stats_df(anom_df, pairs_df):
    rows = []
    for pair in pairs_df['pair'].tolist():
        dfp = anom_df[anom_df['pair'] == pair].copy()
        # distinct timestamps (as minute)
        ts = dfp['timestamp'].drop_duplicates().sort_values()
        count_ts = ts.shape[0]
        # check consecutive minutes: compute diffs in seconds, see if any diff == 60
        consecutive_flag = False
        if count_ts >= 2:
            diffs = ts.diff().dt.total_seconds().dropna()
            if (diffs == 60).any():
                consecutive_flag = True
        rows.append({'pair': pair, 'count_timestamps': count_ts, 'consecutive_flag': 'yes' if consecutive_flag else 'no'})
    return pd.DataFrame(rows)

pair_stats = pair_stats_df(anomalies, pair_counts)

# Keep outputs compact: anomalies_result (top 20), pair_counts, earliest_by_pair, pair_stats
anomalies_result, pair_counts, earliest_by_pair, pair_stats

```
Out[5]:
```
Summary (plain English)

What I did
- Examined trace telemetry inside the incident window 2024-01-25 02:22:51 → 2024-01-25 02:52:51 UTC.
- Kept only inter-service trace rows with trace_name patterns "to_ts-..." or "from_ts-..." that involve any of these services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.
- Used global thresholds computed on the full trace series before filtering: P95 for non-row_count traces and P5 for row_count traces.
- Marked anomalies: row_count anomalies if value <= P5 (low-side), other traces anomalous if value >= P95 (high-side).

Key findings (compact)
- Many inter-service calls between these services were anomalous across the window. Representative anomalous rows (top examples, times UTC):
  - travel-service → ts-travel-service, trace.from_ts-travel-service.row_count at 2024-01-25 02:25:00: value 22 ≤ P5 994.4 (low traffic).
  - ts-travel-service → travel-service, trace.to_ts-travel-service.row_count at 2024-01-25 02:25:00: value 22 ≤ P5 994.4 (low traffic).
  - ts-train-service → train-service, trace.to_ts-train-service.row_count at 2024-01-25 02:25:00: value 272 ≤ P5 2246.4 (low traffic).
  - ts-route-service → route-service, trace.to_ts-route-service.row_count at 2024-01-25 02:25:00: value 248 ≤ P5 1837.6 (low traffic).
  - Many error_rate traces were also flagged (often because global P95 = 0 and values = 0 satisfy value >= P95 — see notes below).

Anomaly counts by service-pair (top results)
- The analysis shows many (source -> dest) pairs with a high number of anomaly rows; many pairs have ~31 anomaly rows across the window (e.g., auth-service -> ts-auth-service; order-service -> ts-order-service; route-service -> ts-route-service; train-service -> ts-train-service; travel_service related pairs).
- Distinct anomaly timestamps per pair tend to be high (examples: 25 distinct minutes for many pairs), and most pairs show consecutive anomalies (consecutive_flag = yes) — i.e., anomalies persist minute-to-minute rather than isolated spikes.

Earliest anomalies
- For almost all (source -> dest) pairs the earliest detected anomaly occurs at 2024-01-25 02:25:00 UTC (a couple of pairs earliest at 02:26:00). So the anomalous behavior begins around 02:25 UTC.

Important interpretation points
- Row_count anomalies are low-side (calls/rows far below historical P5), suggesting decreased inter-service call volume for many pairs around 02:25 onward.
- Many error_rate and other non-row_count traces were flagged partly because their historical P95 is 0; rows with value == 0 therefore met value >= P95 and were labeled anomalous. This is a threshold artifact and should be treated carefully.
- Overall pattern: inter-service call volumes dropped (row_count <= P5) starting ~02:25 UTC and persistent anomalies were observed across many service-to-service pairs for the remainder of the window.

Recommended next steps
- Investigate around 02:25 UTC: check routing, upstream traffic sources, load balancer and network logs to find why call volumes dropped across many service pairs.
- Examine application logs and traces for the high-count pairs (e.g., auth↔ts-auth, order↔ts-order, route↔ts-route, train↔ts-train, travel↔ts-travel) at the earliest anomaly times.
- For error_rate and other zero-valued series, adjust anomaly rules (e.g., ignore series with P95 == 0 or require > P95) to avoid false positives from zero-only historical baselines.

If you want, I can return:
- The exact table of up to 20 anomalous inter-service rows (source, dest, trace_name, timestamp, value, p95, p5),
- The top 20 (source->dest) counts,
- Earliest anomaly times per pair,
- And the per-pair distinct timestamp counts + consecutive-flag — in compact tabular form.

The original code execution output of IPython Kernel is also provided below for reference:

(               source               dest                               trace_name                 timestamp  value     p95      p5
0        auth-service    ts-auth-service    trace.from_ts-auth-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
1   ts-travel-service     travel-service     trace.to_ts-travel-service.row_count 2024-01-25 02:25:00+00:00   22.0  1903.6   994.4
2   ts-travel-service     travel-service    trace.to_ts-travel-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
3   ts-travel-service      train-service      trace.to_ts-train-service.row_count 2024-01-25 02:25:00+00:00    5.0   257.4   142.4
4   ts-travel-service      train-service     trace.to_ts-train-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
5   ts-travel-service      route-service     trace.to_ts-route-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
6      travel-service  ts-travel-service   trace.from_ts-travel-service.row_count 2024-01-25 02:25:00+00:00   22.0  1903.6   994.4
7      travel-service  ts-travel-service  trace.from_ts-travel-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
8    ts-train-service      train-service      trace.to_ts-train-service.row_count 2024-01-25 02:25:00+00:00  272.0  2833.6  2246.4
9    ts-train-service      train-service     trace.to_ts-train-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
10     travel-service   ts-train-service   trace.from_ts-travel-service.row_count 2024-01-25 02:25:00+00:00    5.0   257.4   142.4
11     travel-service   ts-train-service  trace.from_ts-travel-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
12      train-service   ts-train-service    trace.from_ts-train-service.row_count 2024-01-25 02:25:00+00:00  272.0  2833.6  2246.4
13      train-service   ts-train-service   trace.from_ts-train-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
14   ts-route-service      route-service      trace.to_ts-route-service.row_count 2024-01-25 02:25:00+00:00  248.0  2258.0  1837.6
15   ts-route-service      route-service     trace.to_ts-route-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
16  ts-travel-service      route-service      trace.to_ts-route-service.row_count 2024-01-25 02:25:00+00:00    9.0   351.6   210.6
17     travel-service   ts-route-service  trace.from_ts-travel-service.error_rate 2024-01-25 02:25:00+00:00    0.0     0.0     NaN
18     travel-service   ts-route-service   trace.from_ts-travel-service.row_count 2024-01-25 02:25:00+00:00    9.0   351.6   210.6
19       auth-service    ts-auth-service     trace.from_ts-auth-service.row_count 2024-01-25 02:25:00+00:00   54.0   362.0   284.4,                                    pair  count
0       auth-service -> ts-auth-service     31
1     order-service -> ts-order-service     31
2     route-service -> ts-route-service     31
3     train-service -> ts-train-service     31
4    travel-service -> ts-route-service     31
5    travel-service -> ts-train-service     31
6   travel-service -> ts-travel-service     31
7       ts-auth-service -> auth-service     31
8    ts-travel-service -> route-service     31
9     ts-order-service -> order-service     31
10    ts-route-service -> route-service     31
11    ts-train-service -> train-service     31
12  ts-travel-service -> travel-service     31
13   ts-travel-service -> train-service     31
14   travel-service -> ts-order-service     30
15   ts-travel-service -> order-service     30,                                    pair          earliest_anomaly
0       auth-service -> ts-auth-service 2024-01-25 02:25:00+00:00
1     order-service -> ts-order-service 2024-01-25 02:25:00+00:00
2     route-service -> ts-route-service 2024-01-25 02:25:00+00:00
3     train-service -> ts-train-service 2024-01-25 02:25:00+00:00
4    travel-service -> ts-route-service 2024-01-25 02:25:00+00:00
5    travel-service -> ts-train-service 2024-01-25 02:25:00+00:00
6   travel-service -> ts-travel-service 2024-01-25 02:25:00+00:00
7       ts-auth-service -> auth-service 2024-01-25 02:25:00+00:00
8     ts-order-service -> order-service 2024-01-25 02:25:00+00:00
9     ts-route-service -> route-service 2024-01-25 02:25:00+00:00
10    ts-train-service -> train-service 2024-01-25 02:25:00+00:00
11   ts-travel-service -> route-service 2024-01-25 02:25:00+00:00
12   ts-travel-service -> train-service 2024-01-25 02:25:00+00:00
13  ts-travel-service -> travel-service 2024-01-25 02:25:00+00:00
14   travel-service -> ts-order-service 2024-01-25 02:26:00+00:00
15   ts-travel-service -> order-service 2024-01-25 02:26:00+00:00,                                    pair  count_timestamps consecutive_flag
0       auth-service -> ts-auth-service                25              yes
1     order-service -> ts-order-service                25              yes
2     route-service -> ts-route-service                25              yes
3     train-service -> ts-train-service                25              yes
4    travel-service -> ts-route-service                25              yes
5    travel-service -> ts-train-service                25              yes
6   travel-service -> ts-travel-service                25              yes
7       ts-auth-service -> auth-service                25              yes
8    ts-travel-service -> route-service                25              yes
9     ts-order-service -> order-service                25              yes
10    ts-route-service -> route-service                25              yes
11    ts-train-service -> train-service                25              yes
12  ts-travel-service -> travel-service                25              yes
13   ts-travel-service -> train-service                25              yes
14   travel-service -> ts-order-service                24              yes
15   ts-travel-service -> order-service                24              yes)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# 1) Ensure timestamps parsed as UTC datetimes and compute global P95 per (cmdb_id, kpi_name)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

targets = [
    'ts-auth-service',
    'ts-order-service',
    'ts-route-service',
    'ts-train-service',
    'ts-travel-service'
]

mask_targets_full = metric_df['cmdb_id'].isin(targets)

thresholds_p95 = (
    metric_df[mask_targets_full]
    .groupby(['cmdb_id', 'kpi_name'], sort=False)['value']
    .quantile(0.95)
    .reset_index(name='p95')
)

# 2) Filter metric rows to the incident window (inclusive)
start_ts = pd.Timestamp("2024-01-25 02:22:51", tz='UTC')
end_ts   = pd.Timestamp("2024-01-25 02:52:51", tz='UTC')

window_mask = (
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts) &
    metric_df['cmdb_id'].isin(targets)
)
window_df = metric_df[window_mask].copy()

# 3) For each (cmdb_id, kpi_name), mark anomaly points where value >= P95 (use thresholds computed above)
merged = window_df.merge(thresholds_p95, on=['cmdb_id','kpi_name'], how='left')

# Only consider rows where a p95 exists
merged = merged[merged['p95'].notna()].copy()

# Mark anomalies where value >= p95
merged['is_anomaly'] = merged['value'] >= merged['p95']

anomalies = merged[merged['is_anomaly']].copy()

# 4) Group anomaly points into consecutive-minute sequences per (cmdb_id, kpi_name)
# Sort and compute diffs
anomalies = anomalies.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)
anomalies['diff_seconds'] = anomalies.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().dt.total_seconds()
anomalies['start_new'] = anomalies['diff_seconds'].isna() | (anomalies['diff_seconds'] != 60)
anomalies['seq_id'] = anomalies.groupby(['cmdb_id','kpi_name'])['start_new'].cumsum()

# Aggregate sequences
seq_agg = (
    anomalies
    .groupby(['cmdb_id','kpi_name','seq_id'], as_index=False)
    .agg(
        start_timestamp=('timestamp','min'),
        end_timestamp=('timestamp','max'),
        length_minutes=('timestamp','count'),
        max_value=('value','max'),
        p95=('p95','first')
    )
)

# Compute severity_ratio (NaN if p95 <= 0)
seq_agg['severity_ratio'] = np.where(seq_agg['p95'] > 0, seq_agg['max_value'] / seq_agg['p95'], np.nan)

# 5) Filter sequences with length_minutes >=2 AND severity_ratio >= 1.5 (requires p95>0)
result_sequences = seq_agg[
    (seq_agg['length_minutes'] >= 2) &
    (seq_agg['severity_ratio'] >= 1.5)
].copy()

# Sort by length desc then severity_ratio desc and limit to top 20
result_sequences = result_sequences.sort_values(['length_minutes','severity_ratio'], ascending=[False, False]).head(20).reset_index(drop=True)

# Keep output compact: selected columns and head(20)
result_sequences[['cmdb_id','kpi_name','start_timestamp','end_timestamp','length_minutes','max_value','p95','severity_ratio']].head(20)

```
Out[6]:
```
```python
# Summarize the previously computed sustained metric sequences in plain English.
# Reuse in-memory `result_sequences` if available; otherwise produce a fallback message.

try:
    df = result_sequences[['cmdb_id','kpi_name','start_timestamp','end_timestamp','length_minutes','max_value','p95','severity_ratio']]
except NameError:
    df = None

if df is None or df.shape[0] == 0:
    summary = (
        "No sustained, significant metric sequences (>=2 minutes and severity_ratio >= 1.5) "
        "were detected for the specified components within the incident window."
    )
else:
    # Format the top sequences (there is one in the current result)
    rows = []
    for _, r in df.iterrows():
        rows.append(
            f"- {r['cmdb_id']}, KPI '{r['kpi_name']}': {r['start_timestamp']} → {r['end_timestamp']} "
            f"({int(r['length_minutes'])} minutes), max={r['max_value']:.3f}, P95={r['p95']:.3f}, "
            f"severity_ratio={r['severity_ratio']:.3f}"
        )
    details = "\n".join(rows)
    summary = (
        "Sustained, significant metric fault(s) found within the incident window for the specified components:\n\n"
        f"{details}\n\n"
        "Interpretation & next step: The above indicates a sustained CPU surge on ts-order-service "
        "lasting 2 minutes with severity_ratio ≈ 1.63. Investigate ts-order-service CPU causes around 2024-01-25 02:28–02:29 UTC "
        "and correlate with traces, logs, and workload (requests) for root-cause localization."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name           start_timestamp             end_timestamp  length_minutes  max_value       p95  severity_ratio
0  ts-order-service      cpu 2024-01-25 02:28:00+00:00 2024-01-25 02:29:00+00:00               2   7.408021  4.550005        1.628135```
```